In [1208]:
from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import Annotated

import numpy as np

from geneticengine.grammar.metahandlers.ints import IntRange
from geneticengine.grammar import extract_grammar
from geneticengine.problems import SingleObjectiveProblem, MultiObjectiveProblem
from geneticengine.random.sources import NativeRandomSource
from geneticengine.algorithms.gp.gp import GeneticProgramming
from geneticengine.evaluation.budget import TimeBudget, EvaluationBudget
from geneticengine.representations.tree.initializations import MaxDepthDecider
from geneticengine.representations.tree.treebased import TreeBasedRepresentation
from geneticengine.evaluation.recorder import CSVSearchRecorder
from geneticengine.evaluation.tracker import ProgressTracker

from sklearn.datasets import load_breast_cancer

import pandas as pd

from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

import time

In [1209]:
# df = pd.read_csv('fraud_dataset.csv')
#read breast cancer dataset
df = load_breast_cancer(as_frame=True).frame
df.head(3)

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,radius error,texture error,perimeter error,area error,smoothness error,compactness error,concavity error,concave points error,symmetry error,fractal dimension error,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.8,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,1.0950,0.9053,8.589,153.40,0.006399,0.04904,0.05373,0.01587,0.03003,0.006193,25.38,17.33,184.6,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.9,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,0.5435,0.7339,3.398,74.08,0.005225,0.01308,0.01860,0.01340,0.01389,0.003532,24.99,23.41,158.8,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.0,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,0.7456,0.7869,4.585,94.03,0.006150,0.04006,0.03832,0.02058,0.02250,0.004571,23.57,25.53,152.5,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0


In [1210]:
split_percentage = 0.8
split_point = int(len(df) * split_percentage)

# 3. Create the train and test sets
train_df = df.iloc[:split_point]
test_df = df.iloc[split_point:]

# 4. Separate features (X) and target (y) for each set
# Assuming 'is_fraud' is your target column
X_train = train_df.drop('target', axis=1)
y_train = train_df['target']
X_test = test_df.drop('target', axis=1)
y_test = test_df['target']

In [1211]:
feature_names = X_train.columns.tolist()
n_features = len(feature_names)

In [1212]:
class Value(ABC):
    def evaluate(self):
        pass

class Scalar(ABC):
    pass

class Vectorial(ABC):
    pass

In [1213]:
@dataclass
class PastValues(Vectorial):
    index: Annotated[int, IntRange(0, n_features-1)]
    lockback: Annotated[int, IntRange(5, 10)]

    def evaluate(self, X, i):
        start_index = max(0, i-self.lockback+1)
        values = X.iloc[start_index: i+1, self.index].values
        if len(values) < self.lockback: #if there are not enough past values
            padding = np.zeros(self.lockback - len(values))
            values = np.concatenate((padding, values))
        return values
    def __str__(self):
        return f"past_values({feature_names[self.index]}, {self.lockback})"

In [1214]:
@dataclass
class Mean(Scalar):
    arr: Vectorial
    def evaluate(self, X, i):
        v = self.arr.evaluate(X,i)
        return np.mean(v)
    def __str__(self):
        return f"mean({self.arr})"
    
@dataclass
class Max(Scalar):
    arr: Vectorial
    def evaluate(self, X, i):
        v = self.arr.evaluate(X,i)
        return np.max(v)
    def __str__(self):
        return f"max({self.arr})"
    
@dataclass
class Min(Scalar):
    arr: Vectorial
    def evaluate(self, X, i):
        v = self.arr.evaluate(X,i)
        return np.min(v)
    def __str__(self):
        return f"min({self.arr})"

In [1215]:
@dataclass #Scalar Features (1)
class ScalarVar(Scalar): 
    index: Annotated[int, IntRange(0,n_features-1)]

    def evaluate(self, X, i):
        return X.iloc[i, self.index]
    
    def __str__(self):
        return feature_names[self.index]

# @dataclass #Vectorial Features ([1,2,3,4])
# class VectorialVar(Vectorial):
#     index: Annotated[int, IntRange(1,2)]
    
#     def evaluate(self, X):
#         return X[self.index]

In [1216]:
#scalar -> scalar
@dataclass 
class Add(Scalar):
    right: Scalar
    left: Scalar

    def evaluate(self, X, i):
        return self.left.evaluate(X,i) + self.right.evaluate(X,i)
    
    def __str__(self):
        return f"({self.left} + {self.right})"

In [1217]:
grammar = extract_grammar([PastValues, Mean, Max, Min, ScalarVar, Add], Scalar)
print(f"Grammar: {repr(grammar)}")

Grammar: Grammar<Starting=Scalar,Productions={
Scalar -> Mean(arr: Vectorial)|
	Max(arr: Vectorial)|
	Min(arr: Vectorial)|
	ScalarVar(index: Annotated[int])|
	Add(right: Scalar, left: Scalar)

Vectorial -> PastValues(index: Annotated[int], lockback: Annotated[int])
}


In [ ]:
def fitness_function(individual: Value):    
    start = time.perf_counter()

    train_feature = [individual.evaluate(X_train, i) for i in range(len(X_train))]
    test_feature = [individual.evaluate(X_test, i) for i in range(len(X_test))]
    
    X_train_new = np.array(train_feature).reshape(-1,1)
    X_test_new = np.array(test_feature).reshape(-1,1)

    model = DecisionTreeClassifier(random_state=42)
    model.fit(X_train_new, y_train)
    y_pred = model.predict_proba(X_test_new)[:, 1]

    elapsed = time.perf_counter() - start

    return float(elapsed), float(roc_auc_score(y_test, y_pred))

In [ ]:
prob = MultiObjectiveProblem(
    fitness_function=fitness_function,
    minimize=[False, True],
)
r = NativeRandomSource(123)
alg = GeneticProgramming(
    problem=prob,
    budget=TimeBudget(30),
    population_size=50,
    representation=TreeBasedRepresentation(grammar, MaxDepthDecider(r, grammar, 5)),
    random=r,
    tracker=ProgressTracker(
        prob,
        recorders=[CSVSearchRecorder(
            csv_path='outpupt.csv', 
            problem=prob, 
            fields={"Eval Time": lambda t,i,p: i.get_fitness(p).fitness_components[0],
                    "AUC": lambda t,i,p: i.get_fitness(p).fitness_components[1],
                    "Expression": lambda t, i, p: i.get_phenotype(),
                    },
            only_record_best_individuals=False)]
    )
    
)

solutions = alg.search()

In [1223]:
best_sol = max(solutions, key=lambda row: row.get_fitness(prob).fitness_components[1])
print(best_sol.get_phenotype(), best_sol.get_fitness(prob))

((worst smoothness + ((radius error + worst concavity) + (worst concavity + mean symmetry))) + min(past_values(mean concave points, 10))) [0.07256349999806844,0.9217657342657343]


In [1222]:
test = Max(arr=PastValues(index=feature_names.index('worst fractal dimension'), lockback=4))

generated_value = [test.evaluate(X_test, i) for i in range(len(X_test))]

test_df = X_test.copy()
test_df['generated'] = generated_value
test_df['target_x'] = y_test
test_df.head(10)

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,radius error,texture error,perimeter error,area error,smoothness error,compactness error,concavity error,concave points error,symmetry error,fractal dimension error,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,generated,target_x
455,13.380,30.72,86.34,557.2,0.09245,0.07426,0.02819,0.03264,0.1375,0.06016,0.3408,1.9240,2.287,28.93,0.005841,0.012460,0.007936,0.009128,0.01564,0.002985,15.05,41.61,96.69,705.6,0.1172,0.1421,0.07003,0.07763,0.2196,0.07675,0.07675,1
456,11.630,29.29,74.87,415.1,0.09357,0.08574,0.07160,0.02017,0.1799,0.06166,0.3135,2.4260,2.150,23.13,0.009861,0.024180,0.042750,0.009215,0.02475,0.002128,13.12,38.81,86.04,527.8,0.1406,0.2031,0.29230,0.06835,0.2884,0.07220,0.07675,1
457,13.210,25.25,84.10,537.9,0.08791,0.05205,0.02772,0.02068,0.1619,0.05584,0.2084,1.3500,1.314,17.58,0.005768,0.008082,0.015100,0.006451,0.01347,0.001828,14.35,34.23,91.29,632.9,0.1289,0.1063,0.13900,0.06005,0.2444,0.06788,0.07675,1
458,13.000,25.13,82.61,520.2,0.08369,0.05073,0.01206,0.01762,0.1667,0.05449,0.2621,1.2320,1.657,21.19,0.006054,0.008974,0.005681,0.006336,0.01215,0.001514,14.34,31.88,91.06,628.5,0.1218,0.1093,0.04462,0.05921,0.2306,0.06291,0.07675,1
459,9.755,28.20,61.68,290.9,0.07984,0.04626,0.01541,0.01043,0.1621,0.05952,0.1781,1.6870,1.243,11.28,0.006588,0.012700,0.014500,0.006104,0.01574,0.002268,10.67,36.92,68.03,349.9,0.1110,0.1109,0.07190,0.04866,0.2321,0.07211,0.07220,1
460,17.080,27.15,111.20,930.9,0.09898,0.11100,0.10070,0.06431,0.1793,0.06281,0.9291,1.1520,6.051,115.20,0.008740,0.022190,0.027210,0.014580,0.02045,0.004417,22.96,34.49,152.10,1648.0,0.1600,0.2444,0.26390,0.15550,0.3010,0.09060,0.09060,0
461,27.420,26.27,186.90,2501.0,0.10840,0.19880,0.36350,0.16890,0.2061,0.05623,2.5470,1.3060,18.650,542.20,0.007650,0.053740,0.080550,0.025980,0.01697,0.004558,36.04,31.37,251.20,4254.0,0.1357,0.4256,0.68330,0.26250,0.2641,0.07427,0.09060,0
462,14.400,26.99,92.25,646.1,0.06995,0.05223,0.03476,0.01737,0.1707,0.05433,0.2315,0.9112,1.727,20.52,0.005356,0.016790,0.019710,0.006370,0.01414,0.001892,15.40,31.98,100.40,734.6,0.1017,0.1460,0.14720,0.05563,0.2345,0.06464,0.09060,1
463,11.600,18.36,73.88,412.7,0.08508,0.05855,0.03367,0.01777,0.1516,0.05859,0.1816,0.7656,1.303,12.89,0.006709,0.017010,0.020800,0.007497,0.02124,0.002768,12.77,24.02,82.68,495.1,0.1342,0.1808,0.18600,0.08288,0.3210,0.07863,0.09060,1
464,13.170,18.22,84.28,537.3,0.07466,0.05994,0.04859,0.02870,0.1454,0.05549,0.2023,0.6850,1.236,16.89,0.005969,0.014930,0.015640,0.008463,0.01093,0.001672,14.90,23.89,95.10,687.6,0.1282,0.1965,0.18760,0.10450,0.2235,0.06925,0.07863,1
